# Tutorial 6: SBI Surrogate Learning (`sbi_npe`)

Estimated time: 30-50 minutes

## Prerequisites — install BEFORE running this notebook

| Need | Why | Install command |
|---|---|---|
| `bayesian-metamodeling` + `[tutorials]` | Framework runtime + jupyter/matplotlib/numpy. Same as every tutorial. | `pip install -e ".[tutorials]"` |
| **`sbi` + `torch`** | **T6 only.** SBI trains the neural density estimator; without these, Step 2 will skip with a preflight banner. | `pip install -e ".[sbi]"` *or* `conda install -c conda-forge pytorch sbi` |
| `pymc` **as well**, for Steps 4-7 | Steps 4-7 put this notebook's neural surrogate and Tutorial 5's `pymc_gp` surrogate side by side **in one kernel**. With only one backend installed they skip. | Already there if you built the default env: `conda env create -f environment.yml` |

Recommended: complete Tutorial 5 first, **in the same environment**. If T5's artifact is
missing, Step 4 fits it for you — but T5 is where its behaviour is explained.

**If you don't want to install `sbi` right now**, that's fine: the bootstrap cell below detects this and prints a multi-line preflight banner explaining what to install, in which env, and which steps still teach. You can come back when you want to do the full comparison.

(See Tutorial 0 for the full framework-vs-tutorial-vs-backend dependency matrix.)

## Learning aims
- Primary package aim: fit and evaluate an SBI backend through the same user-facing surrogate interface as `pymc_gp`.
- Secondary scientific aim: know what this backend's neural density estimator actually estimates (`p(y | a, b)` — the *forward* map, not the inverse problem NPE is famous for), what its predictive width does and does not mean, and how the choice of backend decides *which failure mode you get*.
- Third aim: meet `log_prob`, the one surrogate method the metamodel layer of T7 actually calls.

## Success criteria
- you can run SBI fit/eval, compare it against `pymc_gp` on matched inputs both inside and outside the training box, and explain why the comparison on this toy is rigged (or, if SBI is not installed, you've read the preflight banner and understand what would happen).

## Why this tutorial matters

*SBI* stands for **simulation-based inference** — a family of methods that learn a probability distribution from simulator runs alone, without needing the simulator's equations in a form you can differentiate. `sbi_npe` is one of them: a *neural posterior estimator*, meaning the distribution it learns is represented by a small neural network.

T5 and T6 fit the *same* toy from the *same* sweep and hand you the same `(mean, std)`
per query point. Everything else about them differs, and the difference is the lesson.

**What `pymc_gp` actually is.** Despite the name there is no Gaussian process in this
framework — no kernel, no covariance function, no `pm.gp` anywhere in `src/`.
`fit_backend_model` dispatches `pymc_gp` to `_fit_pymc_bayesian_linear`
(`src/bayesian_metamodeling/surrogates/backends.py`), which is a Bayesian **linear
regression**:

```
intercept ~ Normal(0, 2)
beta      ~ Normal(0, 2)          # one weight per input
sigma     ~ HalfNormal(1)
y         ~ Normal(intercept + x @ beta, sigma)
```

Three parameters, sampled with NUTS. The backend name is a misnomer that outlived its
original plan — read the code, not the label. Everything you predict about this backend
should follow from *"it is a plane in `a` and `b`"*, because that is all it can be.

**What `sbi_npe` actually is.** A masked autoregressive flow — a stack of invertible
neural transforms — trained on the same `(input, output)` pairs to approximate a
conditional density. No functional form is assumed; the shape is learned. Tens of
thousands of parameters instead of three.

So the contrast that matters is **rigid model class vs flexible density estimator**, not
"GP vs neural":

| | `pymc_gp` | `sbi_npe` |
|---|---|---|
| model class | planes in `(a, b)` | whatever a flow can represent |
| parameters | 3 | tens of thousands |
| truth inside the class | exact, width collapses to ~0 | close, with residual training width |
| truth outside the class | biased — and the width says so (Step 6) | can bend to fit, given data |
| outside the training box | keeps extrapolating its plane, confidently (Step 5) | no guarantee whatsoever |

This tutorial fits SBI on the toy T5 used and asks the question that motivates having
both backends at all: **do they agree?** If they do, you have two independent
confirmations. If they don't, the disagreement is data — it says at least one model
class is wrong for the region you asked about.

## Step 1: Ensure the training dataset exists

The same 3x3 grid sweep T1 and T5 ran, into the same store (`tmp/tutorials/toy_store`).
Both surrogates train on it, and that shared store is what makes the comparison
controlled rather than anecdotal.

**One thing to watch, because it moves every number below.** A run store is
**cumulative by design**: `bayesmm run` writes each sweep under a fresh
`sweeps/<run_id>/`, and `load_tabular_dataset` concatenates **every** `sweep_rows.csv`
under the store root. That is exactly right for a multi-model project sharing one store
(this is how `projects/tcr_signaling` is laid out) — but it means re-running the cell
below *appends* another 9 rows rather than replacing them, and the surrogate trains on
the lot. The cell prints how many rows it will actually see. If that is not 9, this is
why; `rm -rf tmp/tutorials/toy_store` gives you a clean comparison. A store being
cumulative is a feature, and it is a footgun the moment your DOE is not idempotent.

In [ ]:
# Cross-platform setup (Windows / macOS / Linux) — no shell, no PYTHONPATH prefix.
# Find the repo root so `src/` is importable, then load the shared tutorial helpers.
import importlib.util
import os
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "src" / "bayesian_metamodeling").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from bayesian_metamodeling.tutorial import bootstrap, run_mm_cli, run_tool

root = bootstrap()  # chdir to repo root + ensure src/ on sys.path (idempotent)
ROOT = root
print("Repo root:", root)

# Preflight: detect SBI + torch. T6 invokes `bayesmm surrogate fit/eval` with
# the sbi_npe backend, which requires both `sbi` and `torch` in this kernel's
# env. Detecting their absence here lets us skip the SBI-specific steps
# cleanly with an actionable banner instead of a mid-notebook RuntimeError.
SBI_AVAILABLE = (
    importlib.util.find_spec("sbi") is not None
    and importlib.util.find_spec("torch") is not None
)

if not SBI_AVAILABLE:
    _conda_env_name = os.environ.get("CONDA_DEFAULT_ENV")
    BANNER = "=" * 72
    print()
    print(BANNER)
    print("  PREFLIGHT: SBI backend missing — Step 2 onwards will be SKIPPED")
    print(BANNER)
    print(f"  Kernel env path  : {sys.prefix}")
    if _conda_env_name:
        print(f"  Conda env name   : {_conda_env_name}")
    print("  Missing packages : 'sbi' and/or 'torch'")
    print()
    print("  Step 1 (the toy DOE run) still completes — that's the same loop")
    print("  T1 used. Everything from Step 2 on (the SBI fit, the plot, the")
    print("  PyMC-vs-SBI comparisons in Steps 4-7, and the optional appendix)")
    print("  needs SBI.")
    print()
    print("  HOW TO INSTALL — pip works regardless of conda/venv:")
    print()
    print("    # In a terminal, with THIS notebook's kernel env activated")
    if _conda_env_name:
        print(f"    # (your env: '{_conda_env_name}'):")
        print(f"    conda activate {_conda_env_name}")
    else:
        print("    # (activate it first — `conda info --envs` lists conda envs):")
    print("    pip install -e \".[sbi]\"          # via the package's extras")
    print("    # or directly:")
    print("    pip install sbi torch")
    print("    # conda alternative (sbi/torch ARE on conda-forge):")
    print("    conda install -c conda-forge pytorch sbi")
    print()
    print("  After installing, RESTART the Jupyter kernel and re-run from the top.")
    print(BANNER)

# Steps 4-7 put the SBI surrogate next to T5's pymc_gp surrogate on the same query
# points, so those cells need pymc as well as sbi. Detected separately: an sbi-only
# env should still get Steps 1-3 and skip only the comparisons.
PYMC_AVAILABLE = importlib.util.find_spec("pymc") is not None
if SBI_AVAILABLE and not PYMC_AVAILABLE:
    print()
    print("=" * 72)
    print("  NOTE: PyMC missing — Steps 4-7 (the backend comparisons) will SKIP")
    print("=" * 72)
    print("  Steps 1-3 (the SBI surrogate itself) run normally.")
    print("  The comparisons need BOTH backends in one kernel:")
    print("    conda env create -f environment.yml")
    print("    conda activate py314_bayesmm      # has sbi AND pymc")
    print("=" * 72)
    print()

In [ ]:
run_mm_cli('run', 'tutorials/specs/model.toy.grid.json')

# How many rows will the surrogate actually train on? Not necessarily 9 — see above.
import json as _json

import numpy as np

from bayesian_metamodeling.spec import SurrogateSpec
from bayesian_metamodeling.surrogates.dataset import load_tabular_dataset

_count_spec = SurrogateSpec.model_validate(
    _json.loads((root / "tutorials/specs/surrogate.toy.sbi_npe.json").read_text())
)
_x_train, _y_train, _ = load_tabular_dataset(_count_spec)
N_TRAIN = len(_x_train)
N_DISTINCT = len(np.unique(_x_train, axis=0))
print()
print(f"rows the surrogate will train on : {N_TRAIN}")
print(f"distinct design points among them: {N_DISTINCT}")
if N_TRAIN > N_DISTINCT:
    print(
        f"-> this store already holds {N_TRAIN // N_DISTINCT} sweeps of the same grid, so every\n"
        f"   design point is repeated. Duplicated rows add no information but they do\n"
        f"   sharpen both fits, so your widths will be smaller than a clean 9-row run's.\n"
        f"   `rm -rf tmp/tutorials/toy_store` and re-run this cell for the clean version."
    )

## Step 2: Fit and evaluate the SBI surrogate

The whole notebook rests on one spec file, so read it before you run it — the cell below
prints it. Four fields are load-bearing:

- **`density_estimator: "maf"`** — masked autoregressive flow. This is the density
  *family*; the validator also accepts `nsf` and `mdn`.
- **`max_num_epochs: 80` / `stop_after_epochs: 20`** — training budget and
  early-stopping patience. If validation loss stops improving for 20 epochs training
  stops early; otherwise it stops at 80 whether or not the flow converged. That cap is
  why the *"network has not yet fully converged"* warning in the troubleshooting table
  exists.
- **`summary_samples: 256`** — how many draws the reported `mean` and `std` are computed
  from. *This*, not `--n`, sets the precision of every number below.
- **`summary_config: {kind: "index", index: 0}`** — the toy emits `y = [a + b, a * b]`.
  Index `0` selects the sum. Step 6 flips it to `1` and surrogates the product instead;
  watch what that does to each backend.

`training_batch_size: 16` alongside `validation_fraction: 0.1` is worth noticing too:
with a clean 9-row store, one validation row leaves an 8-row training split, so a
"batch" is the entire split. With the store larger than that it becomes a real
minibatch — another way the cumulative store changes the fit.

In [ ]:
if not SBI_AVAILABLE:
    print("Step 2 SKIPPED (SBI/torch missing in this kernel) — see preflight banner above.")
else:
    print("--- tutorials/specs/surrogate.toy.sbi_npe.json ---")
    print((root / "tutorials/specs/surrogate.toy.sbi_npe.json").read_text())
    run_mm_cli("surrogate", "fit", "tutorials/specs/surrogate.toy.sbi_npe.json")
    run_mm_cli(
        "surrogate", "eval", "tutorials/specs/surrogate.toy.sbi_npe.json",
        "--inputs", '{"a":[0.25,0.75,1.25,1.75],"b":[0.2,0.6,1.0,1.4]}',
        "--n", "200",
    )

### Reading that JSON — and the trap in `--n`

- **`artifact_id`** — which fitted model answered. `eval` resolves it by `spec_name`, so
  the most recent fit for this spec shadows every earlier one.
- **`sample_shape: [4, 200]`** — 4 query rows x 200 raw draws.
- **`samples_preview`** — the first few of those draws, so you can see they really are
  draws from a distribution and not a point estimate repeated.
- **`summary.mean` / `summary.std`** — the two numbers everything below is built on.

**`--n` does not do what it looks like it does.** In `eval_surrogate`
(`src/bayesian_metamodeling/surrogates/service.py`) `n` is forwarded only to
`model.sample(...)`. `model.summary(inputs)` takes no `n` at all: it draws
`backend_config.summary_samples` samples (256 here) at a fixed seed. So `--n 2` and
`--n 300` return **byte-identical** `summary.mean` and `summary.std`. If you want a less
noisy summary, raise `summary_samples` in the spec; turning up `--n` only lengthens the
preview. (The self-check at the bottom asserts this, so the notebook would notice if it
ever stopped being true.)

### What the flow actually learned — the one thing to get right

In textbook SBI, `theta` is the simulator's **parameters**, `x` is **observed data**, and
NPE learns `p(theta | x)`: the inverse problem — given data, which parameters produced
it?

**This framework wires it the other way round.** In `_fit_sbi_npe`
(`src/bayesian_metamodeling/surrogates/backends.py`) the training call is

```python
inference.append_simulations(theta=y_norm, x=x_norm)
```

with `theta` = the model **output** `y`, and `x` = the model **inputs** `(a, b)`. So what
this backend learns is

```
p(y | a, b)
```

— the **forward** map. NPE is being used purely as an amortized conditional density
estimator: an emulator. Nothing about that direction is inherent to NPE; it is a choice
this backend made, and it is precisely why `sbi_npe` and `pymc_gp` are interchangeable
behind one spec at all — both answer *"what `y` at this `(a, b)`?"*. If you know some SBI
and expected this surrogate to infer `(a, b)` from an observed `y`, it does the opposite.

**And there is no prior object.** `_build_sbi_inference` constructs `NPE` with no prior
(the legacy `SNPE` path passes `prior=None` explicitly), so sbi auto-derives support from
the training `theta` — which is exactly why the backend has to suppress sbi's *"The
passed prior has no support property"* warning. What plays the prior's role is the 3x3
DOE grid over `(a, b)`, but as the distribution of the **conditioning** variable: the flow
is only trained to be accurate where the DOE went. Hold onto that until Step 5.

Both backends hand you `(mean, std)` per query point. Those two numbers mean different
things in the two backends, and Step 4 is where that stops being an abstraction.

## Step 3: Plot the SBI predictive summary (graphic)

Four query points the surrogate never saw, plotted against `a + b` so the analytical
truth is the diagonal, with the training observations on the same axes. The table
underneath is the part to read carefully: the last column puts each error in units of
the surrogate's own standard deviation.

In [ ]:
if not SBI_AVAILABLE:
    print("Step 3 SKIPPED (SBI/torch missing) — see preflight banner above.")
    sbi_mean = None
    sbi_std = None
    sbi_inputs = None
else:
    import csv
    import json

    import matplotlib.pyplot as plt
    import numpy as np

    from bayesian_metamodeling.spec import SurrogateSpec
    from bayesian_metamodeling.surrogates import eval_surrogate

    # `root` came from bootstrap() at the top of the notebook — no path autodetect here.
    spec_payload = json.loads((root / "tutorials/specs/surrogate.toy.sbi_npe.json").read_text())
    spec = SurrogateSpec.model_validate(spec_payload)
    inputs = {"a": [0.25, 0.75, 1.25, 1.75], "b": [0.2, 0.6, 1.0, 1.4]}
    result = eval_surrogate(spec=spec, inputs_payload=inputs, n=300)
    mean = np.asarray(result["summary"]["mean"], dtype=float)
    std = np.asarray(result["summary"].get("std", [0.0] * len(mean)), dtype=float)

    query_x = np.asarray(inputs["a"]) + np.asarray(inputs["b"])
    truth = query_x  # the toy's y__0 is exactly a + b

    # The training observations, straight out of the sweep CSVs the fit itself read.
    train_x, train_y = [], []
    for _csv_path in sorted((root / "tmp/tutorials/toy_store/sweeps").rglob("sweep_rows.csv")):
        with open(_csv_path) as _fh:
            for _row in csv.DictReader(_fh):
                if _row.get("status") != "success":
                    continue
                train_x.append(float(_row["a"]) + float(_row["b"]))
                train_y.append(float(_row["y__0"]))  # y__0 = a + b; y__1 = a * b

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    xx = np.linspace(0, 4, 50)
    ax.plot(xx, xx, "k:", alpha=0.5, label="analytical truth: y = a + b")
    ax.scatter(
        train_x, train_y, marker="x", s=70, c="tab:gray", alpha=0.6, zorder=2,
        label=f"training observations (N={len(train_x)})",
    )
    ax.errorbar(
        query_x, mean, yerr=std, fmt="o", color="tab:orange", markersize=9, capsize=5,
        zorder=3, label=f"SBI NPE, mean ± std (mean std = {std.mean():.1e})",
    )
    ax.set_title("SBI surrogate at 4 query points it never saw (toy: y = a + b)")
    ax.set_xlabel("a + b")
    ax.set_ylabel("y")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.show()

    err = np.abs(mean - truth)
    print(f"{'a+b':>6} {'truth':>8} {'mean':>10} {'std':>11} {'|error|':>10} {'error/std':>10}")
    for _xv, _tv, _mv, _sv, _ev in zip(query_x, truth, mean, std, err):
        print(f"{_xv:6.2f} {_tv:8.2f} {_mv:10.4f} {_sv:11.3e} {_ev:10.4f} {_ev / _sv:10.1f}")
    print()
    print("The flow reproduced a function it was never given the formula for, from")
    print(f"{len(train_x)} examples. Now read the last column: it is the error measured in")
    print(f"units of the surrogate's own std (largest here: {np.max(err / std):.1f}). Where that")
    print("exceeds 1, the prediction is wrong by more than the width it reports. This std")
    print("is a residual TRAINING-noise width, not a calibrated error bar.")

    # Stash predictions for the comparison cells below.
    sbi_mean = mean
    sbi_std = std
    sbi_inputs = inputs

## Predict before you look

You now have two surrogates trained on identical data: T5's `pymc_gp` (a Bayesian plane)
and this notebook's normalizing flow. Commit to answers **before** running the next
cells — the exercise is worth nothing once you have seen the tables.

1. **Inside** the `[0, 2]²` box the DOE covered: will they largely agree, or clearly
   disagree?
2. **Outside** it, at `a = b = 5` (truth `y = 10`): whose mean stays closest to the
   truth? Whose error bar *widens* to warn you that you have left the training region?
3. If the toy emitted `y = a · b` instead of `y = a + b`, which backend would win, and
   why?

Step 4 answers (1). Step 5 answers (2) — and one half of the usual answer is simply
false here. Step 6 answers (3).

**Why this comparison is the lesson:** choosing a surrogate backend is not a performance
question, it is a question of *which failure mode you can live with* — and of whether
you would notice it.

## Step 4: Compare the two backends on the same query points

Both surrogates read the same store, so this is controlled: identical training data,
identical query points, two model classes. If T5's `pymc_gp` artifact is not present in
this environment, the cell fits it first (that is T5's Step 2).

In [ ]:
if not SBI_AVAILABLE or sbi_mean is None or not PYMC_AVAILABLE:
    # Name the backend that is ACTUALLY missing. An earlier version printed
    # "install SBI" in both branches, so a reader in `py312_bayesmm_sbi` — the env
    # this very notebook tells them to use — was advised to install the one thing
    # they already had.
    if not PYMC_AVAILABLE and SBI_AVAILABLE:
        print("Step 4 SKIPPED — SBI is present, PyMC is not.")
        print("This comparison loads T5's pymc_gp surrogate, so it needs BOTH backends")
        print("in one kernel. Steps 1-3 above ran for real.")
    elif not SBI_AVAILABLE and PYMC_AVAILABLE:
        print("Step 4 SKIPPED — PyMC is present, SBI is not.")
        print("Steps 2-3 need SBI, so there are no NPE predictions to compare against.")
    else:
        print("Step 4 SKIPPED — needs both backends; this kernel has neither.")
    print()
    print("To run it:  conda env create -f environment.yml")
    print("            conda activate py314_bayesmm     # has PyMC *and* SBI")
    print("Then re-run this notebook. Steps 4-7 are the only place in the series where")
    print("the two surrogate families are put side by side, so it is worth the extra env.")
else:
    import json

    import matplotlib.pyplot as plt
    import numpy as np

    from bayesian_metamodeling.spec import SurrogateSpec
    from bayesian_metamodeling.storage.surrogate_store import find_latest_artifact_for_spec
    from bayesian_metamodeling.surrogates import eval_surrogate, fit_surrogate

    pymc_spec_payload = json.loads((root / "tutorials/specs/surrogate.toy.pymc_gp.json").read_text())
    pymc_spec = SurrogateSpec.model_validate(pymc_spec_payload)

    # T5 normally produces this artifact. If this env never ran T5, fit it now so the
    # comparison is still available — same spec, same store, same 4 query points.
    try:
        find_latest_artifact_for_spec(pymc_spec.name)
    except ValueError:
        print(f"No artifact for spec '{pymc_spec.name}' here — fitting it (this is T5's Step 2).")
        fit_surrogate(pymc_spec)

    pymc_result = eval_surrogate(spec=pymc_spec, inputs_payload=sbi_inputs, n=300)
    pymc_mean = np.asarray(pymc_result["summary"]["mean"], dtype=float)
    pymc_std = np.asarray(pymc_result["summary"].get("std", [0.0] * len(pymc_mean)), dtype=float)

    # Plot both predictive series on shared axes, with `a + b` (the truth) as x.
    query_x = np.asarray(sbi_inputs["a"]) + np.asarray(sbi_inputs["b"])
    truth_x = np.linspace(0, 4, 50)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(truth_x, truth_x, "k:", alpha=0.5, label="analytical truth: y = a + b")
    ax.errorbar(
        query_x - 0.04, pymc_mean, yerr=pymc_std, fmt="o", color="tab:blue",
        markersize=10, capsize=5, label=f"pymc_gp (mean ± std={pymc_std.mean():.1e})",
    )
    ax.errorbar(
        query_x + 0.04, sbi_mean, yerr=sbi_std, fmt="s", color="tab:orange",
        markersize=10, capsize=5, label=f"sbi_npe (mean ± std={sbi_std.mean():.1e})",
    )
    ax.set_title("pymc_gp vs sbi_npe on the same query points (toy: y = a + b)")
    ax.set_xlabel("a + b")
    ax.set_ylabel("y")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.show()

    print(f"{'a+b':>6} | {'pymc mean':>10} {'pymc std':>10} | {'sbi mean':>10} {'sbi std':>10}")
    for _i, _xv in enumerate(query_x):
        print(
            f"{_xv:6.2f} | {pymc_mean[_i]:10.4f} {pymc_std[_i]:10.2e} |"
            f" {sbi_mean[_i]:10.4f} {sbi_std[_i]:10.2e}"
        )
    print()
    print(f"mean |error| : pymc_gp {np.mean(np.abs(pymc_mean - query_x)):.5f}"
          f"   sbi_npe {np.mean(np.abs(sbi_mean - query_x)):.5f}")
    print(f"widths differ by a factor of {sbi_std.mean() / pymc_std.mean():.0f}"
          " — same units, different provenance (read on).")

## Mini-lesson: how to read the comparison

The means nearly coincide; the widths differ by the factor the cell just printed. Those
widths are not the same kind of object:

- **`pymc_gp`'s width** is posterior uncertainty about **three parameters**. Training
  points that lie exactly on a plane pin an intercept and two slopes down to numerical
  precision, so the width collapses to the ~1e-6 scale printed above. It is small
  because the model class *contains* the truth and the data identifies it — not because
  this backend is "better".
- **`sbi_npe`'s width** is residual training error of a density estimator with four
  orders of magnitude more parameters. It stops shrinking when early stopping fires and
  it will never reach zero. That is the price of not writing a likelihood down.

Same units, entirely different provenance. You cannot read them as two answers to *"how
sure am I?"*.

**The lesson is what to do when they disagree.** On THIS toy they barely disagree on the
mean. But on a realistic biological model you might see `pymc_gp` say `0.4 ± 0.05` and
`sbi_npe` say `0.6 ± 0.3`. That disagreement is data:

- If the tight band sits *inside* the wide one → the wide one is just less confident;
  defer to the tight one **where you believe its model class**.
- If the two means are *well separated* (say >2 std apart) → at least one surrogate's
  assumptions are wrong in this region. Don't trust either prediction without re-running
  the simulator at that input.
- If both are confident and they disagree → both are extrapolating, and differently.
  Usually this means there is no nearby training data and the surrogates are guessing.

**When to reach for which.** `sbi_npe` earns its cost when the response is not close to
anything you can write down: multi-modal outputs, discrete branches, no closed-form
likelihood. `pymc_gp` wins whenever a plane is a defensible description — and it wins
*loudly*: faster to fit, orders of magnitude faster to query (Step 7 times it on your
machine), and its width tells you when the plane is wrong (Step 6). Have both;
cross-check on every coupled-model task.

## Step 5: Leave the box

Every number so far came from inside `[0, 2]²`, where the DOE sampled. Nothing you have
seen says anything about the outside, and the only honest move is to look.

The cell below queries both surrogates at `(0.5, 0.5)` — inside — and at `(5, 5)` and
`(-3, -3)`, far outside, where no training point came near. `err/std` is again the error
in units of the surrogate's own reported width: it is the column that tells you whether
the uncertainty was any use.

In [ ]:
if not SBI_AVAILABLE or sbi_mean is None or not PYMC_AVAILABLE:
    print("Step 5 SKIPPED — needs both backends in one kernel (see Step 4).")
else:
    import numpy as np

    from bayesian_metamodeling.surrogates import eval_surrogate

    out_inputs = {"a": [0.5, 5.0, -3.0], "b": [0.5, 5.0, -3.0]}
    out_truth = np.asarray(out_inputs["a"]) + np.asarray(out_inputs["b"])
    out = {}
    for _tag, _sp in (("pymc_gp", pymc_spec), ("sbi_npe", spec)):
        _r = eval_surrogate(spec=_sp, inputs_payload=out_inputs, n=50)
        out[_tag] = (
            np.asarray(_r["summary"]["mean"], dtype=float),
            np.asarray(_r["summary"]["std"], dtype=float),
        )

    print("DOE support for both inputs was [0.0, 2.0] (see model.toy.grid.json)\n")
    print(
        f"{'(a, b)':>12} {'truth':>7} | {'pymc mean':>10} {'std':>10} {'err/std':>9}"
        f" | {'sbi mean':>10} {'std':>10} {'err/std':>9}"
    )
    for _i, (_a, _b) in enumerate(zip(out_inputs["a"], out_inputs["b"])):
        _pm, _ps = out["pymc_gp"][0][_i], out["pymc_gp"][1][_i]
        _sm, _ss = out["sbi_npe"][0][_i], out["sbi_npe"][1][_i]
        _where = "inside" if abs(_a) <= 2.0 else "OUTSIDE"
        print(
            f"{f'({_a}, {_b})':>12} {out_truth[_i]:7.2f} |"
            f" {_pm:10.4f} {_ps:10.2e} {abs(_pm - out_truth[_i]) / _ps:9.1f} |"
            f" {_sm:10.4f} {_ss:10.2e} {abs(_sm - out_truth[_i]) / _ss:9.1f}   <- {_where}"
        )
    print()
    print("Compare each backend's std inside vs outside the box before reading the answers.")

### Answers to the prediction exercise

Check these against what you wrote down.

**1. Inside the box: they agree**, to a few decimal places on the mean. Both model
classes contain (or can bend to) a plane, and the DOE surrounds every query point.

**2. Outside the box, the popular answer is wrong.** `pymc_gp` does **not** revert to a
prior mean, and its band does **not** widen: a plane extrapolates its plane forever, so
at `(5, 5)` it returns 10 with a width still of order 1e-6 — no meaningful widening at
all. Here
that happens to be *exactly right* — because the truth **is** a plane. On any nonlinear
system the identical behaviour would be confidently, catastrophically wrong, and nothing
in the output would warn you. That is more dangerous than a model that widens and
complains.

`sbi_npe` is badly wrong out there too, and its width barely moves either — read its
`err/std` column.

**So the lesson is not "one of them degrades gracefully".** It is that a small `std`
means *"consistent with what I was trained on"* and never *"close to the truth"*, and
**neither backend has any way to tell you that you left the box**. Detecting that is
your job: compare the query against the DOE support declared in the ModelSpec
(`support: [0.0, 2.0]` for both inputs here) before you believe any surrogate. This is
the answer to the question T1 left you holding, and it is less comfortable than it
sounded.

**3.** Step 6, next.

## Step 6: Why this comparison is rigged

Step 4 looked like a win for `pymc_gp`. It was — but not because it is the better
backend. It won because `y = a + b` is a plane and `pymc_gp`'s entire model class is
planes. **The winner of a surrogate comparison is decided by which model class contains
the truth, not by which backend is better engineered.**

You can show that with one edit. The toy already emits `y = [a + b, a * b]`, and
`summary_config.index` picks which element to surrogate. Flip it to `1` and the target
becomes `y = a · b` — a product, which no plane in `a` and `b` can represent. Nothing
else about either spec changes; the cell below edits that one field in memory, refits
both backends on the same store, and re-queries the same four points.

Predict first: what happens to `pymc_gp`'s **mean**, and — separately — what happens to
its **std**?

In [ ]:
if not SBI_AVAILABLE or not PYMC_AVAILABLE:
    print("Step 6 SKIPPED — needs both backends in one kernel (see Step 4).")
else:
    import json

    import numpy as np

    from bayesian_metamodeling.spec import SurrogateSpec
    from bayesian_metamodeling.surrogates import eval_surrogate, fit_surrogate

    prod_q = {"a": [0.25, 0.75, 1.25, 1.75], "b": [0.2, 0.6, 1.0, 1.4]}
    prod_truth = np.asarray(prod_q["a"]) * np.asarray(prod_q["b"])
    print(f"target is now y = a * b; truth at the query points: "
          f"{[round(float(t), 3) for t in prod_truth]}\n")

    prod_rows = {}
    for _fname in ("surrogate.toy.pymc_gp.json", "surrogate.toy.sbi_npe.json"):
        _payload = json.loads((root / "tutorials/specs" / _fname).read_text())
        _payload["name"] = _payload["name"] + "_product"       # its own artifact lineage
        _payload["summary_config"] = {"kind": "index", "index": 1}   # <- the only real edit
        _prod_spec = SurrogateSpec.model_validate(_payload)
        fit_surrogate(_prod_spec)
        _r = eval_surrogate(spec=_prod_spec, inputs_payload=prod_q, n=50)
        prod_rows[_prod_spec.backend] = (
            np.asarray(_r["summary"]["mean"], dtype=float),
            np.asarray(_r["summary"]["std"], dtype=float),
        )

    print(f"{'backend':>9} | {'mean predictions':>36} | {'mean std':>10} {'MAE':>8} {'max err/std':>12}")
    for _be, (_m, _s) in prod_rows.items():
        _pretty = "[" + ", ".join(f"{v:6.3f}" for v in _m) + "]"
        print(
            f"{_be:>9} | {_pretty:>36} | {_s.mean():10.2e}"
            f" {np.mean(np.abs(_m - prod_truth)):8.4f} {np.max(np.abs(_m - prod_truth) / _s):12.1f}"
        )
    print(f"\n{'truth':>9} | " + "[" + ", ".join(f"{v:6.3f}" for v in prod_truth) + "]")

The `pymc_gp` row is the interesting one. Its **mean** is now biased — a plane pushed
through a saddle, best-fitting and still wrong. And unlike Step 5, its **std** does not
stay at the 1e-6 scale: it jumps by orders of magnitude, to something the size of the
residuals themselves. That is the honest half of a rigid model class: when the truth is
outside it, `sigma` absorbs the misfit and the width says so — check `pymc_gp`'s
`max err/std` column: it is now of order 1, meaning the width it reports has grown to
cover the error it actually makes. That is what an honest error bar looks like, and it
is the first time in this notebook you have seen one.

Put Steps 5 and 6 next to each other, because they are the two distinct failure modes:

| | query far from data | truth outside the model class |
|---|---|---|
| `pymc_gp` width | does **not** widen (Step 5) | **does** widen (Step 6) |
| what that means | you get no warning | you do get a warning |

The flow does much better on the mean here — a product is well within what it can
represent. But look at its `max err/std` column: it is still off by multiples of its own
width. Better mean, still not a calibrated error bar.

**The ranking flipped when the target changed, and neither backend's `std` told you which
regime you were in.** A surrogate comparison on one function is not a backend benchmark;
it is a statement about that function. When you pick a backend for a real model, the
question to ask is not "which is more accurate?" but "what shape do I believe the
response has, and what happens if I am wrong about that?".

## Step 7: Where this artifact goes next

You have used two of the three methods on the `SurrogateModel` protocol: `sample` and
`summary`. **The metamodel layer uses neither.** It calls **`log_prob`**, and that call
is the only contact point between the metamodel layer and the surrogate layer.

`log_prob(inputs, outputs)` answers: *how surprised is this surrogate by the claim that
`y` takes this value at these inputs?* In T7 that number becomes one factor in a joint
density over several models' variables — the surrogate's vote on a proposed value. A
surrogate whose `log_prob` did not clearly prefer the truth would be useless there,
however good its `mean` looked.

Two things to carry into T7:

**1. The default metamodel command never evaluates your surrogate.** `bayesmm meta
sample` defaults to `--method propagate`, which draws every variable from its prior,
overwrites each coupled target with `transform(source)`, and calls the compiled model
with `surrogates={}` (`meta/sampling.py`). The artifact you just fitted contributes
nothing; the written `inference_data.json` even records `"surrogates_evaluated": false`.
Only `--method joint` (`meta/joint_sampling.py`) puts `log_prob` into the density, and
only there do both ends of a coupling move. **If you fit a surrogate and your metamodel
output looks exactly like the prior, this is why.**

**2. Backend choice has a runtime price, and `log_prob` is where you pay it.** Joint
sampling is a random-walk Metropolis chain — one `log_prob` per surrogate per proposal —
so the ratio the cell below measures multiplies straight into your wall clock.

In [ ]:
if not SBI_AVAILABLE or not PYMC_AVAILABLE:
    print("Step 7 SKIPPED — needs both backends in one kernel (see Step 4).")
else:
    import json
    import time
    from pathlib import Path

    import numpy as np

    from bayesian_metamodeling.spec import SurrogateSpec
    from bayesian_metamodeling.storage.surrogate_store import find_latest_artifact_for_spec
    from bayesian_metamodeling.surrogates.backends import load_backend_model

    probe = {"a": np.array([1.0]), "b": np.array([0.5])}   # truth here: y = a + b = 1.5
    ms_per_call = {}
    for _fname in ("surrogate.toy.pymc_gp.json", "surrogate.toy.sbi_npe.json"):
        _sp = SurrogateSpec.model_validate(
            json.loads((root / "tutorials/specs" / _fname).read_text())
        )
        _aid, _apath = find_latest_artifact_for_spec(_sp.name)
        _art = json.loads(_apath.read_text())
        _model = load_backend_model(
            _sp.backend, Path(_art["backend_payload"]),
            expected_inputs=_sp.inputs, expected_outputs=_sp.outputs,
        )
        _lp_true = float(np.asarray(_model.log_prob(probe, {"y": np.array([1.5])})).reshape(-1)[0])
        _lp_off = float(np.asarray(_model.log_prob(probe, {"y": np.array([3.5])})).reshape(-1)[0])
        _t0 = time.perf_counter()
        for _ in range(10):
            _model.log_prob(probe, {"y": np.array([1.5])})
        ms_per_call[_sp.backend] = (time.perf_counter() - _t0) * 100.0
        print(
            f"{_sp.backend:>8}: log_prob(y=1.5, the truth) = {_lp_true:>12.3f}"
            f"   log_prob(y=3.5) = {_lp_off:>18.3f}"
            f"   {ms_per_call[_sp.backend]:>7.2f} ms/call"
        )
    print()
    print(f"sbi_npe's log_prob costs {ms_per_call['sbi_npe'] / ms_per_call['pymc_gp']:.0f}x "
          "what pymc_gp's does on this machine.")
    print("Both prefer the truth by a huge margin — that is what makes them usable as")
    print("likelihood factors at all. The cost gap is what decides whether joint sampling")
    print("finishes in seconds or in minutes; budget for it, and prefer fewer, longer chains.")

No notebook in this series ever feeds an `sbi_npe` surrogate into a metamodel — T7 uses
prebuilt example artifacts, T8 and T9 use `pymc_gp`. The runnable proof that it does work
is `tests/test_joint_sampling_backends.py`, which drives joint sampling through both
backends and is where the timing figures above come from.

### One field is not enough: what "swappable" actually means

The recap is about to claim that backends are swappable behind one spec contract. Test
it the obvious way first — take T5's `pymc_gp` spec, change `backend` to `sbi_npe`, and
validate.

In [ ]:
import json

from bayesian_metamodeling.spec import SurrogateSpec

payload = json.loads((root / "tutorials/specs/surrogate.toy.pymc_gp.json").read_text())
payload["backend"] = "sbi_npe"          # the only edit
try:
    SurrogateSpec.model_validate(payload)
    print("validated — which is NOT what should happen; see the note below.")
except Exception as exc:                 # pydantic wraps the spec validator's ValueError
    print(f"rejected by the spec validator ({type(exc).__name__}):\n")
    print(str(exc).split("For further information")[0].strip())

`backend_config` is **not** backend-neutral, and `validate_backend_config`
(`src/bayesian_metamodeling/surrogate_config.py`) refuses to pretend otherwise:
`draws`, `tune`, `chains` and `target_accept` configure a NUTS sampler and mean nothing
to a normalizing flow. Silently ignoring them would be worse — you would think you had
set something.

What *is* neutral is everything else: `inputs`, `outputs`, `dataset_ref`,
`summary_config`, `seed`, and the whole downstream contract (artifact -> `sample` /
`log_prob` / `summary`). That is the abstraction, and it is why the two shipped tutorial
specs differ only in `name`, `backend` and `backend_config`, and why every cell above
could call `eval_surrogate` without caring which backend answered.

## Recap: what T6 established

- **The spec contract is backend-neutral; `backend_config` is not.** `inputs`,
  `outputs`, `dataset_ref`, `summary_config`, `seed` and the artifact ->
  `sample`/`log_prob`/`summary` contract carry over untouched. Hyperparameters belong to
  a backend, and the validator makes you say which one you meant.
- **`pymc_gp` is a Bayesian linear regression, not a Gaussian process** — three
  parameters, no kernel. Read `_fit_pymc_bayesian_linear`, not the name. **`sbi_npe`
  learns `p(y | a, b)`** — the *forward* map, NPE machinery used as an emulator, not the
  inverse problem it is famous for.
- **A small `std` means "consistent with my training distribution", never "close to the
  truth".** Step 5 put both backends many multiples of their own width away from the
  answer, with no widening to warn you.
- **Two different failure modes, only one of them visible.** A rigid model class widens
  when its *class* is wrong (Step 6) and stays confident when the *query* is far away
  (Step 5). Knowing which one you are in is your job, not the surrogate's.
- **Which backend "wins" is a property of the target function.** Flipping
  `summary_config.index` from the sum to the product flipped the ranking, with no change
  to either backend.
- **`log_prob` is what the metamodel layer consumes** — and `meta sample`'s default
  `--method propagate` never calls it. Only `--method joint` conditions on your
  surrogate.

One sentence to carry forward: *the spec contract is the abstraction — but a surrogate's
error bar is only as trustworthy as the region and the model class it came from.*

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `Step 4 SKIPPED — SBI is present, PyMC is not` (also Steps 5-7) | Only one backend in this kernel; the comparisons need both | `conda env create -f environment.yml`, then re-run this notebook on the `py314_bayesmm` kernel. No single-backend env can run them. |
| `PREFLIGHT: SBI backend missing` | `sbi` and/or `torch` absent from this kernel | `conda env create -f environment.yml` (or `environment-sbi.yml` if you only want Steps 1-3), then switch kernel. |
| `Maximum number of epochs reached, but network has not yet fully converged` | Training hit its `max_num_epochs` cap before early stopping fired | Expected on tiny data. Note the framework pins `sbi<0.27` — on 0.27 this warning appears routinely and, under `filterwarnings = error`, becomes a hard test failure. |
| The numbers differ from a previous run of this notebook | Re-running Step 1 appends **another** sweep under `tmp/tutorials/toy_store/sweeps/`, and the surrogate trains on *every* sweep in the store — so the training set grew | `rm -rf tmp/tutorials/toy_store`, then re-run from Step 1. The seed is already set (`seed: 123`; `_fit_sbi_npe` calls `torch.manual_seed`) and training is reproducible *given a fixed dataset*. What changed was the dataset, not the sampler. |
| Predictions are confidently wrong outside the training range | Neither backend detects that you left the DOE box | Not a bug — Step 5 is that lesson. Check the query against `io_schema.inputs[*].support` yourself. A `pymc_gp` width of 1e-6 at `a = 100` is not evidence of anything. |
| `No surrogate artifact found for spec_name=tutorial_toy_surrogate_pymc` | T5 was never run in this environment | Step 4 fits it for you; if you skipped Step 4, run Tutorial 5 in this kernel. |
| Import succeeds but fitting errors on a kwarg | sbi changed its API between versions | The backend carries two compat layers (`tracker=` vs legacy `summary_writer=`). Versions outside 0.22-0.26 are untested. |

**Next:** T7 couples two surrogates into one metamodel — and the method it uses decides
whether the artifact you just fitted is consulted at all (Step 7).

## Optional appendix: confidence check that the SBI backend works

The cell below runs a single regression test from the package's own test suite. It's not
part of the lesson — just an "is my SBI install actually functional" smoke test you can
run if anything above surprised you. It exercises `fit`, `sample` **and** `log_prob`,
which is the trio Step 7 cares about.

In [ ]:
if not SBI_AVAILABLE:
    print("Optional appendix SKIPPED (SBI missing in this kernel).")
else:
    run_tool(
        "pytest", "-q", "tests/test_surrogate_backends.py",
        "-k", "sbi_npe_backend_fit_sample_and_logprob",
        check=False,
    )

## Final check: T6's SBI surrogate ran (or skipped cleanly)

If `SBI_AVAILABLE`, five assertions — each able to fail on a notebook that *ran* but
taught nothing:

1. a surrogate artifact exists **for this notebook's spec name**, not merely some
   `sbi_npe` artifact left in `tmp/` by the test suite (that weaker check passed even
   when Step 2 never ran);
2. the predictive **mean** is within absolute error 0.5 of `y = a + b` — a
   predict-the-training-mean surrogate scores ~0.9 and fails this;
3. the predictive **width** is small but non-zero: `1e-4 < mean(std) < 0.1`. A width of
   ~0 means the flow collapsed to a delta; above 0.1 means it never converged. Both
   bounds are deliberately loose — the width shrinks as the cumulative store grows, so
   the check is on the *regime*, not on one machine's fit;
4. `summary.posterior_draws` equals the spec's `summary_samples` and not `--n` — the
   claim Step 2 makes about where the summary's precision comes from;
5. `log_prob` prefers the truth at every query point: `log_prob(y = a + b) >
   log_prob(y = a + b + 2)`. That is the method the metamodel layer calls, and nothing
   else in this notebook would notice if it broke.

Otherwise: the preflight banner was printed and Step 2 onwards was deliberately skipped.

In [ ]:
# Self-check: SBI surrogate produced (or preflight skipped cleanly).
import json as _json

import numpy as _np

if not SBI_AVAILABLE:
    print(f"\n[T6 self-check OK] Step 2+ skipped per preflight (SBI_AVAILABLE={SBI_AVAILABLE}).")
else:
    from pathlib import Path as _P

    from bayesian_metamodeling.spec import SurrogateSpec as _SS
    from bayesian_metamodeling.storage.surrogate_store import find_latest_artifact_for_spec as _find
    from bayesian_metamodeling.surrogates import eval_surrogate as _eval
    from bayesian_metamodeling.surrogates.backends import load_backend_model as _load

    _spec = _SS.model_validate(
        _json.loads((root / "tutorials/specs/surrogate.toy.sbi_npe.json").read_text())
    )

    # (1) an artifact for THIS spec. Filtering on backend alone would be satisfied by a
    # stray sbi_npe artifact from the test suite — i.e. it passed even if Step 2 never ran.
    try:
        _aid, _apath = _find(_spec.name)
    except ValueError as _exc:
        raise AssertionError(
            f"No surrogate artifact for spec_name={_spec.name!r} — Step 2's fit didn't run."
        ) from _exc
    _art = _json.loads(_apath.read_text())
    assert _art.get("backend") == "sbi_npe", f"artifact backend is {_art.get('backend')!r}"
    assert _art.get("spec_name") == _spec.name, f"artifact spec_name is {_art.get('spec_name')!r}"

    _inputs = {"a": [0.25, 0.75, 1.25, 1.75], "b": [0.2, 0.6, 1.0, 1.4]}
    _truth = _np.asarray(_inputs["a"]) + _np.asarray(_inputs["b"])
    _result = _eval(spec=_spec, inputs_payload=_inputs, n=200)
    _mean = _np.asarray(_result["summary"]["mean"], dtype=float)
    _std = _np.asarray(_result["summary"]["std"], dtype=float)

    # (2) the mean is a real fit, not the training mean.
    _mae = float(_np.mean(_np.abs(_mean - _truth)))
    assert _mae < 0.5, f"SBI mean prediction MAE={_mae:.4f} > 0.5 — surrogate not converged."

    # (3) the width is in the right REGIME. Loose on purpose: it shrinks as the
    # cumulative store grows, so a tight bound would fail on an honest second run
    # rather than on a broken fit.
    _w = float(_np.mean(_std))
    assert 1e-4 < _w < 0.1, (
        f"SBI predictive width mean={_w:.3e} outside (1e-4, 0.1): ~0 means the flow "
        "collapsed to a delta, >0.1 means it never converged."
    )

    # (4) `--n` must not touch the summary — the claim Step 2 makes.
    _expected_draws = int(_spec.backend_config["summary_samples"])
    assert _result["summary"]["posterior_draws"] == _expected_draws, (
        f"summary.posterior_draws={_result['summary']['posterior_draws']} != "
        f"summary_samples={_expected_draws} — `--n` is leaking into the summary."
    )

    # (5) log_prob discriminates — the only surrogate method the metamodel layer calls.
    _model = _load(
        _spec.backend, _P(_art["backend_payload"]),
        expected_inputs=_spec.inputs, expected_outputs=_spec.outputs,
    )
    _q = {_k: _np.asarray(_v, dtype=float) for _k, _v in _inputs.items()}
    _lp_true = _np.asarray(_model.log_prob(_q, {"y": _truth})).reshape(-1)
    _lp_off = _np.asarray(_model.log_prob(_q, {"y": _truth + 2.0})).reshape(-1)
    assert _np.all(_lp_true > _lp_off), (
        f"log_prob does not prefer the truth: {_lp_true.tolist()} vs {_lp_off.tolist()} — "
        "this surrogate would be useless as a metamodel likelihood factor."
    )

    print(
        f"\n[T6 self-check OK] MAE={_mae:.5f} (<0.5); mean predictive width={_w:.3e} "
        f"(in 1e-4..0.1); summary from {_expected_draws} draws (not --n); "
        f"log_prob margin={float(_np.min(_lp_true - _lp_off)):.1f} nats; "
        f"artifact {_apath}"
    )